# Exercise 5: Learning Curves, Hyperparameter Tuning and Evaluation Metrics

The previous exercises each trained a segmentation model on a real dataset. This exercise steps back from any particular satellite application and instead focuses on the tools needed to judge whether a model has trained well and to compare one model against another. These tools are the same regardless of whether the underlying model is a U-Net, a SegFormer or a foundation model such as TerraMind.

To keep training fast enough to repeat many times within this notebook, we use a small synthetic segmentation problem rather than a satellite dataset. Images are generated on the fly and contain circular blobs on a noisy background, playing the role of a rare object of interest, for example a flooded field or a landslide scar, against a much larger background class.

## What you will learn

1. How to read training and validation learning curves to diagnose underfitting and overfitting
2. How accuracy, precision, recall, F1 score and Intersection over Union are each computed, and what each one does and does not tell you
3. Why classification accuracy can be a misleading metric when one class is much rarer than another
4. How to run a small hyperparameter search and compare configurations using validation metrics
5. How the choice of decision threshold trades precision against recall

In [ ]:
!pip install torch scikit-learn matplotlib numpy -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, jaccard_score

SEED = 42
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## A synthetic segmentation task

The function below draws a small number of circular blobs onto a noisy background. Every pixel inside a blob is labelled 1, the object class, and every other pixel is labelled 0, the background class. Because the blobs are small relative to the whole image, the two classes are imbalanced, similar to how flooded pixels or landslide pixels are usually a small fraction of a full satellite scene.

In [ ]:
IMAGE_SIZE = 64


def generate_synthetic_scene(image_size=IMAGE_SIZE, rng=None):
    rng = rng if rng is not None else np.random.RandomState()

    image = rng.normal(loc=0.3, scale=0.15, size=(image_size, image_size)).astype(np.float32)
    mask = np.zeros((image_size, image_size), dtype=np.int64)

    row_grid, col_grid = np.mgrid[0:image_size, 0:image_size]
    number_of_blobs = rng.randint(1, 4)
    for _ in range(number_of_blobs):
        center_row = rng.randint(0, image_size)
        center_col = rng.randint(0, image_size)
        radius = rng.randint(3, 9)
        blob = (row_grid - center_row) ** 2 + (col_grid - center_col) ** 2 <= radius ** 2
        image[blob] += rng.uniform(0.4, 0.6)
        mask[blob] = 1

    image = np.clip(image, 0.0, 1.0)
    return image[np.newaxis, :, :], mask  # add a channel dimension to the image


class SyntheticSegmentationDataset(Dataset):
    """Generates a fixed set of scenes, one per index, so the dataset is the same every epoch."""

    def __init__(self, num_samples, seed):
        self.num_samples = num_samples
        self.seed = seed

    def __len__(self):
        return self.num_samples

    def __getitem__(self, index):
        rng = np.random.RandomState(self.seed + index)
        image, mask = generate_synthetic_scene(rng=rng)
        return torch.from_numpy(image), torch.from_numpy(mask)


train_dataset = SyntheticSegmentationDataset(num_samples=300, seed=0)
val_dataset = SyntheticSegmentationDataset(num_samples=100, seed=10_000)

sample_image, sample_mask = train_dataset[0]
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample_image[0], cmap="gray")
axes[0].set_title("Synthetic scene")
axes[1].imshow(sample_mask, cmap="RdYlGn_r", vmin=0, vmax=1)
axes[1].set_title("Object mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Fraction of object pixels in this sample: {sample_mask.float().mean():.3f}")

## A small model

The purpose of this exercise is to study evaluation rather than architecture, so a very small convolutional network is enough. It applies a few convolutions at the full image resolution and produces two output channels, one score per class for every pixel, without any downsampling or upsampling.

In [ ]:
class TinySegmentationNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=2, hidden_channels=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Conv2d(hidden_channels, num_classes, kernel_size=1)

    def forward(self, x):
        return self.classifier(self.features(x))

## Training with a fixed configuration

We first train the model once with a reasonable set of default settings, and record the training and validation loss and accuracy at every epoch. These recordings are the learning curves that the next section will interpret.

In [ ]:
def run_one_epoch(model, loader, optimizer, criterion, device, training):
    model.train() if training else model.eval()

    total_loss = 0.0
    correct_pixels = 0
    total_pixels = 0

    torch.set_grad_enabled(training)
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        if training:
            optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, masks)

        if training:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        predictions = torch.argmax(logits, dim=1)
        correct_pixels += (predictions == masks).sum().item()
        total_pixels += masks.numel()

    return total_loss / len(loader.dataset), correct_pixels / total_pixels


def train_model(learning_rate=1e-3, class_weight=None, num_epochs=25, batch_size=16):
    model = TinySegmentationNet().to(device)
    weight_tensor = None if class_weight is None else torch.tensor(class_weight, device=device)
    criterion = nn.CrossEntropyLoss(weight=weight_tensor)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}
    for _ in range(num_epochs):
        train_loss, train_accuracy = run_one_epoch(model, train_loader, optimizer, criterion, device, training=True)
        val_loss, val_accuracy = run_one_epoch(model, val_loader, optimizer, criterion, device, training=False)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_accuracy"].append(val_accuracy)

    return model, history


baseline_model, baseline_history = train_model(learning_rate=1e-3, num_epochs=25)

## Reading learning curves

A learning curve plots a measurement, usually loss or accuracy, against training epoch, separately for the training set and the validation set. The shape of these two curves relative to each other is informative.

If both curves stay high and flat, the model is underfitting, meaning it is not able to capture the pattern in the data at all, often because it is too small, trained for too few epochs, or trained with too low a learning rate. If the training curve keeps improving while the validation curve stalls or gets worse, the model is overfitting, meaning it is starting to memorise details specific to the training samples rather than learning a pattern that generalises. If both curves improve together and then flatten out at a similar level, training has likely converged to a reasonable solution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(baseline_history["train_loss"], label="Training loss")
axes[0].plot(baseline_history["val_loss"], label="Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross entropy loss")
axes[0].set_title("Loss curve")
axes[0].legend()

axes[1].plot(baseline_history["train_accuracy"], label="Training accuracy")
axes[1].plot(baseline_history["val_accuracy"], label="Validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Pixel accuracy")
axes[1].set_title("Accuracy curve")
axes[1].legend()

plt.tight_layout()
plt.show()

## Why accuracy alone is not enough

The synthetic object class typically covers only a small fraction of each image. A model that simply predicts background everywhere would already reach a high accuracy score without detecting a single object pixel. This is exactly the situation described above, where a metric looks encouraging while the model has learned little of practical use.

The next cells compute a full confusion matrix, meaning the counts of true positives, false positives, false negatives and true negatives, and use it to derive several metrics by hand. Each metric is then cross checked against the equivalent function from scikit learn, to confirm that the formulas are being applied correctly.

In [ ]:
def collect_predictions(model, loader, device):
    model.eval()
    all_predictions, all_targets = [], []
    with torch.no_grad():
        for images, masks in loader:
            logits = model(images.to(device))
            predictions = torch.argmax(logits, dim=1).cpu().numpy()
            all_predictions.append(predictions.reshape(-1))
            all_targets.append(masks.numpy().reshape(-1))
    return np.concatenate(all_predictions), np.concatenate(all_targets)


def confusion_counts(predictions, targets, positive_class=1):
    predicted_positive = predictions == positive_class
    actual_positive = targets == positive_class

    true_positive = int(np.sum(predicted_positive & actual_positive))
    false_positive = int(np.sum(predicted_positive & ~actual_positive))
    false_negative = int(np.sum(~predicted_positive & actual_positive))
    true_negative = int(np.sum(~predicted_positive & ~actual_positive))
    return true_positive, false_positive, false_negative, true_negative


val_loader_for_metrics = DataLoader(val_dataset, batch_size=16, shuffle=False)
val_predictions, val_targets = collect_predictions(baseline_model, val_loader_for_metrics, device)

tp, fp, fn, tn = confusion_counts(val_predictions, val_targets)
print(f"True positive  : {tp}")
print(f"False positive : {fp}")
print(f"False negative : {fn}")
print(f"True negative  : {tn}")

In [ ]:
def accuracy_manual(tp, fp, fn, tn):
    return (tp + tn) / (tp + fp + fn + tn)


def precision_manual(tp, fp):
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0


def recall_manual(tp, fn):
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0


def f1_manual(precision, recall):
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0


def iou_manual(tp, fp, fn):
    return tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0


manual_accuracy = accuracy_manual(tp, fp, fn, tn)
manual_precision = precision_manual(tp, fp)
manual_recall = recall_manual(tp, fn)
manual_f1 = f1_manual(manual_precision, manual_recall)
manual_iou = iou_manual(tp, fp, fn)

print(f"{'Metric':12s}{'Manual':>10s}{'scikit learn':>14s}")
print(f"{'Accuracy':12s}{manual_accuracy:10.4f}{accuracy_score(val_targets, val_predictions):14.4f}")
print(f"{'Precision':12s}{manual_precision:10.4f}{precision_score(val_targets, val_predictions):14.4f}")
print(f"{'Recall':12s}{manual_recall:10.4f}{recall_score(val_targets, val_predictions):14.4f}")
print(f"{'F1 score':12s}{manual_f1:10.4f}{f1_score(val_targets, val_predictions):14.4f}")
print(f"{'IoU':12s}{manual_iou:10.4f}{jaccard_score(val_targets, val_predictions):14.4f}")

## What each metric captures, and where it falls short

Accuracy is the fraction of all pixels classified correctly. It is easy to understand, but it weighs every pixel equally, so with a rare object class it is dominated by how well the common background class is predicted, and can stay high even when the object class is detected poorly.

Precision is the fraction of pixels predicted as the object class that are actually correct. It answers the question, when the model says object, how often is it right. Precision drops when the model produces many false alarms.

Recall is the fraction of true object pixels that the model successfully finds. It answers the question, of all the real objects, how many did the model detect. Recall drops when the model misses real objects.

Precision and recall usually trade off against each other, since a model can often improve one by sacrificing the other, for example by being more or less willing to predict the object class. F1 score summarises both into a single number using their harmonic mean, which stays low unless precision and recall are both reasonably high. F1 score is useful for comparing models with a single number, but it does not reveal whether a poor score comes from too many false alarms or too many missed detections, so it is best reported alongside precision and recall rather than instead of them.

Intersection over Union, often written IoU, measures the overlap between the predicted object area and the true object area, divided by their combined area. It is closely related to F1 score, and for a single class the two rank models identically, but IoU is the standard metric reported in most segmentation literature and is intuitive to interpret directly as a percentage overlap. Its main limitation is that it can be very sensitive to small objects, since missing or misplacing even a few pixels on a small object changes the ratio much more than the same number of pixels would change it on a large object.

## Hyperparameter tuning

Hyperparameters are settings chosen before training starts, such as the learning rate or how heavily to weight the rare class in the loss function, as opposed to the model weights that training itself adjusts. Different choices can lead to noticeably different results, so it is common practice to train several short runs with different settings and compare them on the validation set before committing to a longer final training run.

The cell below trains several models with different learning rates, and separately, different weightings placed on the object class within the loss function, then compares them using validation IoU and F1 score.

In [ ]:
def evaluate_iou_and_f1(model, loader, device):
    predictions, targets = collect_predictions(model, loader, device)
    tp, fp, fn, tn = confusion_counts(predictions, targets)
    return iou_manual(tp, fp, fn), f1_manual(precision_manual(tp, fp), recall_manual(tp, fn))


learning_rates_to_try = [1e-2, 1e-3, 1e-4]
learning_rate_results = []

for learning_rate in learning_rates_to_try:
    model, _ = train_model(learning_rate=learning_rate, num_epochs=25)
    iou, f1 = evaluate_iou_and_f1(model, val_loader_for_metrics, device)
    learning_rate_results.append((learning_rate, iou, f1))
    print(f"Learning rate {learning_rate:8.0e}  validation IoU {iou:.4f}  validation F1 {f1:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
rates = [str(r[0]) for r in learning_rate_results]
ious = [r[1] for r in learning_rate_results]
f1s = [r[2] for r in learning_rate_results]

x_positions = np.arange(len(rates))
ax.bar(x_positions - 0.2, ious, width=0.4, label="Validation IoU")
ax.bar(x_positions + 0.2, f1s, width=0.4, label="Validation F1")
ax.set_xticks(x_positions)
ax.set_xticklabels(rates)
ax.set_xlabel("Learning rate")
ax.set_ylabel("Score")
ax.set_title("Effect of learning rate on validation performance")
ax.legend()
plt.tight_layout()
plt.show()

The class weighting experiment below repeats the same idea for a different hyperparameter. Weighting the object class more heavily in the loss function pushes the model toward predicting the object class more often, which typically increases recall at some cost to precision. Watching how precision, recall, F1 score and IoU move together as this weight changes is a good illustration of why a single metric rarely tells the whole story.

In [ ]:
object_class_weights_to_try = [1.0, 3.0, 8.0]
weight_results = []

for object_weight in object_class_weights_to_try:
    model, _ = train_model(learning_rate=1e-3, class_weight=[1.0, object_weight], num_epochs=25)
    predictions, targets = collect_predictions(model, val_loader_for_metrics, device)
    tp, fp, fn, tn = confusion_counts(predictions, targets)
    precision = precision_manual(tp, fp)
    recall = recall_manual(tp, fn)
    iou = iou_manual(tp, fp, fn)
    weight_results.append((object_weight, precision, recall, iou))
    print(f"Object class weight {object_weight:4.1f}  "
          f"precision {precision:.4f}  recall {recall:.4f}  IoU {iou:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
weights = [str(w[0]) for w in weight_results]
precisions = [w[1] for w in weight_results]
recalls = [w[2] for w in weight_results]

ax.plot(weights, precisions, marker="o", label="Precision")
ax.plot(weights, recalls, marker="o", label="Recall")
ax.set_xlabel("Object class weight in the loss function")
ax.set_ylabel("Score")
ax.set_title("Effect of class weighting on precision and recall")
ax.legend()
plt.tight_layout()
plt.show()

## The effect of the decision threshold

So far, a pixel has been assigned to the object class whenever the model scores it higher than the background class, which corresponds to a probability threshold of 0.5. This threshold is itself a choice, and shifting it changes precision and recall in a predictable way. Lowering the threshold makes the model label more pixels as the object class, which tends to raise recall and lower precision. Raising the threshold has the opposite effect.

The cell below sweeps the threshold applied to the predicted object probability and plots precision, recall and F1 score as functions of it, which makes it possible to choose a threshold deliberately rather than always defaulting to 0.5.

In [ ]:
def collect_object_probabilities(model, loader, device):
    model.eval()
    all_probabilities, all_targets = [], []
    with torch.no_grad():
        for images, masks in loader:
            logits = model(images.to(device))
            probabilities = torch.softmax(logits, dim=1)[:, 1, :, :]
            all_probabilities.append(probabilities.cpu().numpy().reshape(-1))
            all_targets.append(masks.numpy().reshape(-1))
    return np.concatenate(all_probabilities), np.concatenate(all_targets)


object_probabilities, targets_for_threshold = collect_object_probabilities(baseline_model, val_loader_for_metrics, device)

thresholds = np.linspace(0.05, 0.95, 19)
precisions_by_threshold, recalls_by_threshold, f1s_by_threshold = [], [], []

for threshold in thresholds:
    predictions_at_threshold = (object_probabilities >= threshold).astype(np.int64)
    tp, fp, fn, tn = confusion_counts(predictions_at_threshold, targets_for_threshold)
    precision = precision_manual(tp, fp)
    recall = recall_manual(tp, fn)
    precisions_by_threshold.append(precision)
    recalls_by_threshold.append(recall)
    f1s_by_threshold.append(f1_manual(precision, recall))

best_threshold = thresholds[int(np.argmax(f1s_by_threshold))]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(thresholds, precisions_by_threshold, label="Precision")
ax.plot(thresholds, recalls_by_threshold, label="Recall")
ax.plot(thresholds, f1s_by_threshold, label="F1 score")
ax.axvline(best_threshold, color="grey", linestyle="--", label=f"Best F1 at threshold {best_threshold:.2f}")
ax.set_xlabel("Decision threshold on the object class probability")
ax.set_ylabel("Score")
ax.set_title("Precision, recall and F1 score across decision thresholds")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

Learning curves reveal whether a model is training well before any test time metric is even considered, by comparing training and validation performance over epochs. Accuracy is intuitive but can hide poor performance on a rare class. Precision, recall and F1 score describe correctness with respect to a specific class and its typical trade offs. IoU is the standard way to describe overlap in segmentation tasks and is closely related to F1 score. Hyperparameter tuning is the practice of comparing several training configurations on a validation set, using exactly the metrics discussed here, before selecting a final configuration. The decision threshold applied to model probabilities is itself a tunable choice that shifts the balance between precision and recall.

None of these metrics replace looking directly at predictions, as done in Exercises 2, 3 and 4. A high IoU score is reassuring, but inspecting where a model succeeds and where it fails on real scenes remains an essential part of evaluating a segmentation model.